# Radish Bank: Local MongoDB to Redis Iris Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/redis/redis-iris-demos/blob/main/notebooks/rdi_to_iris_workshop.ipynb)

This is the banking-domain workshop. It uses the existing **`domains/radish-bank`** domain pack, not a separate toy schema.

## Recommended local launch

Run these commands from the repo root before opening the notebook:

```bash
uv sync --extra notebook
uv run --extra notebook jupyter notebook notebooks/rdi_to_iris_workshop.ipynb
```

Use the repo-managed kernel started by `uv`. Avoid running this notebook in a shared/global Python kernel because package versions can conflict with other notebooks.

## What this workshop does

1. Generate the Radish Bank domain models and demo data.
2. Seed the generated banking records into a local MongoDB replica set.
3. Generate MongoDB -> Redis Data Integration (RDI) pipeline files for every Radish Bank entity.
4. Create a Redis Iris Context Surface from the Radish Bank `ContextModel` classes.
5. Either deploy real RDI, or use the notebook lab lane to copy MongoDB records into Redis JSON keys with the same key contract.
6. Run Context Retriever, Agent Memory, LangCache, and a domain-aware chat loop end to end.

Local MongoDB is used because it is the easiest source system to show live. The RDI key contract mirrors the banking domain schema: for example, `Account` uses `radish_bank_account:{account_id}` and the generated RDI job writes that same key.

## Demo UI handoff

The notebook sets up and validates the data path. To show the normal Redis Iris demo UI after the data is in Redis, run the app from a terminal with `DEMO_DOMAIN=radish-bank` and the same Redis/Iris credentials.

# Setup

## Environment Check

Local users should already have installed dependencies with:

```bash
uv sync --extra notebook
uv run --extra notebook jupyter notebook notebooks/rdi_to_iris_workshop.ipynb
```

The next cell verifies that the notebook is running in an environment with the repo dependencies plus the notebook extra. It only installs packages automatically on Colab, where there is no repo-managed `uv` environment.

In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_IMPORTS = {
    "context_surfaces": "context-surfaces",
    "fastapi": "fastapi",
    "httpx": "httpx",
    "langchain_openai": "langchain-openai",
    "langgraph": "langgraph",
    "openai": "openai",
    "pydantic": "pydantic",
    "pydantic_settings": "pydantic-settings",
    "pymongo": "pymongo",
    "redis": "redis",
    "redisvl": "redisvl",
}

if "google.colab" in sys.modules:
    # Colab has no repo-managed uv environment, so install the project dependencies there.
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "context-surfaces>=0.0.1",
        "fastapi>=0.115.0",
        "httpx>=0.28.0",
        "langchain-openai>=1.1.12",
        "langgraph>=0.2.70",
        "langgraph-checkpoint-redis>=0.4.0",
        "openai>=1.68.0",
        "pydantic>=2.10.0",
        "pydantic-settings>=2.7.0",
        "python-dotenv>=1.0.1",
        "redis>=5.2.0",
        "redisvl>=0.16.0",
        "sentence-transformers>=3.0.0",
        "uvicorn[standard]>=0.34.0",
        "pymongo>=4.8.0",
    ])
    print("Colab dependencies installed. Restart the kernel if prompted.")
else:
    missing = [dist for module, dist in REQUIRED_IMPORTS.items() if importlib.util.find_spec(module) is None]
    if missing:
        raise RuntimeError(
            "Missing notebook dependencies: "
            + ", ".join(missing)
            + "\n\nFrom the repo root, run:\n"
            + "  uv sync --extra notebook\n"
            + "  uv run --extra notebook jupyter notebook notebooks/rdi_to_iris_workshop.ipynb\n"
            + "Then reopen this notebook from that Jupyter server."
        )
    print(f"Ready ({sys.executable})")
    if ".venv" not in str(Path(sys.executable)):
        print("Note: this does not look like the repo .venv. If imports fail later, relaunch with uv run --extra notebook jupyter notebook.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk-community 0.2.0 requires redis<6.0.0,>=5.0.0, but you have redis 6.4.0 which is incompatible.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.4.8 which is incompatible.
langchain-experimental 0.3.4 requires langchain-core<0.4.0,>=0.3.28, but you have langchain-core 1.4.8 which is incompatible.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.4.8 which is incompatible.
langgraph-checkpoint-redis 0.1.2 requires langgraph-checkpoint<3.0.0,>=2.0.21, but you have langgraph-checkpoint 4.1.1 which is incompatible.
gradio 5.49.1 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
redis-context-course 1.0.

### Grab workshop files (if Colab)

**Local:** skip the next cell if you already cloned `redis-iris-demos` and opened this notebook from the repo-managed Jupyter server.

The MongoDB cells are intended for local execution with Docker. Colab can display and inspect the notebook, but it is not the recommended runtime for the local MongoDB lane.

In [2]:
# NBVAL_SKIP
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    if not os.path.isdir("redis-iris-demos"):
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/redis/redis-iris-demos.git"])
    os.chdir("redis-iris-demos")
    print("Changed directory to redis-iris-demos")
else:
    print("Local: skipped clone - use your existing repo checkout.")


Local: skipped clone - use your existing repo checkout.


In [3]:
import sys
from pathlib import Path


def _find_notebooks_dir(start: Path) -> Path:
    """Find the notebooks directory that contains workshop_helpers.py."""
    matches = []
    for candidate in [start, *start.parents]:
        if (candidate / "workshop_helpers.py").is_file():
            matches.append(candidate)
        elif (candidate / "notebooks" / "workshop_helpers.py").is_file():
            matches.append(candidate / "notebooks")
    return matches[-1] if matches else start


NOTEBOOKS_DIR = _find_notebooks_dir(Path.cwd())
ROOT = NOTEBOOKS_DIR.parent
for p in (NOTEBOOKS_DIR, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

NOTEBOOKS_DIR

PosixPath('/Users/nitin.kanukolanu/workspace/redis-iris-demos/notebooks')

## Step 1 - Configure credentials

Paste Redis Cloud, Redis Iris, Agent Memory, LangCache, and OpenAI values into `WORKSHOP_CONFIG`.

The local MongoDB source cells can run without these values. Cells that create the Context Surface, write to Redis, seed Memory/LangCache, or run the agent validate the required external values before making calls.

For the demo UI handoff, use the same values in `.env` or export them in the terminal where you run `make dev`.

In [ ]:
import os

DOMAIN_ID = "radish-bank"
MONGODB_URI = "mongodb://localhost:27017/?replicaSet=rs0"
MONGODB_DATABASE = "radish_bank_workshop"

WORKSHOP_CONFIG = {
    "DEMO_DOMAIN": DOMAIN_ID,
    "OPENAI_API_KEY": "",        # OpenAI key - powers chat and embeddings
    "REDIS_HOST": "",            # Redis Cloud hostname used by Context Retriever and the RDI target
    "REDIS_PORT": "",            # Redis Cloud port
    "REDIS_PASSWORD": "",        # Redis Cloud password
    "REDIS_SSL": "true",         # "true" for Redis Cloud TLS
    "CTX_ADMIN_KEY": "",         # Context Retriever admin API key
    "MEMORY_API_BASE_URL": "",   # Agent Memory REST base URL
    "MEMORY_STORE_ID": "",       # Your memory store ID
    "MEMORY_API_KEY": "",        # Agent Memory API key
    "LANGCACHE_HOST": "",        # LangCache server host
    "LANGCACHE_CACHE_ID": "",    # LangCache cache ID
    "LANGCACHE_API_KEY": "",     # LangCache API key
    "OPENAI_CHAT_MODEL": "gpt-4o-mini",
    "REDIS_USERNAME": "default",
    "REDIS_DB": "0",
}

REQUIRED_IRIS_KEYS = (
    "OPENAI_API_KEY",
    "REDIS_HOST",
    "REDIS_PORT",
    "REDIS_PASSWORD",
    "CTX_ADMIN_KEY",
    "MEMORY_API_BASE_URL",
    "MEMORY_STORE_ID",
    "MEMORY_API_KEY",
    "LANGCACHE_HOST",
    "LANGCACHE_CACHE_ID",
    "LANGCACHE_API_KEY",
)


def apply_notebook_config(config: dict[str, str]) -> None:
    for key, value in config.items():
        if value is not None and str(value).strip():
            os.environ[key] = str(value).strip()


def missing_iris_keys() -> list[str]:
    return [key for key in REQUIRED_IRIS_KEYS if not (os.environ.get(key) or "").strip()]


def require_iris_config() -> None:
    missing = missing_iris_keys()
    if missing:
        raise RuntimeError(
            "Missing required Redis Iris/OpenAI credentials in WORKSHOP_CONFIG:\n  - "
            + "\n  - ".join(missing)
        )


apply_notebook_config(WORKSHOP_CONFIG)
print(f"Domain: {DOMAIN_ID}")
print(f"MongoDB source: {MONGODB_URI}")
print(f"MongoDB database: {MONGODB_DATABASE}")
if missing_iris_keys():
    print("External Iris credentials still missing:", ", ".join(missing_iris_keys()))
else:
    print("External Iris credentials are set")

# Part 1 - Validate and generate the Radish Bank domain

These cells use the repo's real banking domain pack:

- `domains/radish-bank/schema.py`
- `domains/radish-bank/generated_models.py`
- `domains/radish-bank/data_generator.py`
- `domains/radish-bank/domain.py`

In [ ]:
import subprocess

# Keep generated ContextModel classes in sync with domains/radish-bank/schema.py.
subprocess.run([sys.executable, str(ROOT / "scripts" / "generate_models.py"), "--domain", DOMAIN_ID], check=True)
subprocess.run([sys.executable, str(ROOT / "scripts" / "validate_domain.py"), "--domain", DOMAIN_ID], check=True)

In [ ]:
import importlib.util
import json
from pathlib import Path

from backend.app.core.domain_loader import load_domain


def _load_module_from_path(module_name: str, path: Path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Unable to load {module_name} from {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


domain = load_domain(DOMAIN_ID)
output_dir = ROOT / domain.manifest.output_dir
generated = domain.generate_demo_data(output_dir=output_dir, update_env_file=False)

entity_specs = domain.get_entity_specs()
generated_models = _load_module_from_path(
    "radish_bank_generated_models_workshop",
    ROOT / domain.manifest.generated_models_path,
)

for key, value in generated.env_updates.items():
    os.environ[key] = str(value).strip('"')

print("Generated Radish Bank data:", generated.output_dir)
print(json.dumps(generated.summary, indent=2))

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


records_by_class = {
    spec.class_name: read_jsonl(output_dir / spec.file_name)
    for spec in entity_specs
}
records_by_collection = {
    Path(spec.file_name).stem: records_by_class[spec.class_name]
    for spec in entity_specs
}
collection_by_class = {
    spec.class_name: Path(spec.file_name).stem
    for spec in entity_specs
}
id_field_by_collection = {
    Path(spec.file_name).stem: spec.id_field
    for spec in entity_specs
}
key_template_by_collection = {
    Path(spec.file_name).stem: spec.redis_key_template
    for spec in entity_specs
}
model_by_class = {
    spec.class_name: getattr(generated_models, spec.class_name)
    for spec in entity_specs
}

for collection_name, rows in records_by_collection.items():
    print(f"{collection_name}: {len(rows)} record(s)")

# Part 2 - Start and seed local MongoDB

RDI's MongoDB source uses change streams/oplog, so the local database must run as a replica set. The repo includes `notebooks/mongo_local/docker-compose.yml` for a single-node replica set.

Run this locally. If you already have MongoDB running with `replicaSet=rs0` on port `27017`, skip the Docker cell and update `MONGODB_URI` if needed.

In [ ]:
import shutil
import subprocess
import time

compose_path = NOTEBOOKS_DIR / "mongo_local" / "docker-compose.yml"
if not compose_path.exists():
    raise FileNotFoundError(f"Missing {compose_path}")
if shutil.which("docker") is None:
    raise RuntimeError("Docker is required for the local MongoDB path. Install Docker or point MONGODB_URI at an existing replica set.")

subprocess.run(["docker", "compose", "-f", str(compose_path), "up", "-d"], check=True)

for _ in range(30):
    ping = subprocess.run(
        ["docker", "exec", "iris-workshop-mongo", "mongosh", "--quiet", "--eval", "db.adminCommand('ping').ok"],
        text=True,
        capture_output=True,
    )
    if ping.returncode == 0 and "1" in ping.stdout:
        break
    time.sleep(1)
else:
    raise RuntimeError("MongoDB container did not become ready")

rs_eval = """
try {
  rs.status().ok
} catch (e) {
  rs.initiate({_id: 'rs0', members: [{_id: 0, host: 'localhost:27017'}]}).ok
}
"""
subprocess.run(
    ["docker", "exec", "iris-workshop-mongo", "mongosh", "--quiet", "--eval", rs_eval],
    check=True,
)
print("Local MongoDB replica set is running")

In [ ]:
from pymongo import MongoClient, ReplaceOne


def mongo_client() -> MongoClient:
    return MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5000)


def seed_mongo_source(records: dict[str, list[dict]]) -> dict[str, int]:
    client = mongo_client()
    try:
        client.admin.command("ping")
        db = client[MONGODB_DATABASE]
        counts: dict[str, int] = {}
        for collection_name, rows in records.items():
            id_field = id_field_by_collection[collection_name]
            collection = db[collection_name]
            collection.delete_many({})
            collection.create_index(id_field, unique=True)
            if rows:
                ops = [ReplaceOne({id_field: row[id_field]}, row, upsert=True) for row in rows]
                collection.bulk_write(ops)
            counts[collection_name] = collection.count_documents({})
        return counts
    finally:
        client.close()


mongo_counts = seed_mongo_source(records_by_collection)
mongo_counts

In [ ]:
def read_mongo_source_rows(collections: list[str] | None = None) -> dict[str, list[dict]]:
    client = mongo_client()
    try:
        db = client[MONGODB_DATABASE]
        selected = collections or list(records_by_collection)
        output: dict[str, list[dict]] = {}
        for collection_name in selected:
            id_field = id_field_by_collection[collection_name]
            rows = []
            for row in db[collection_name].find({}, {"_id": 0}).sort(id_field):
                rows.append(dict(row))
            output[collection_name] = rows
        return output
    finally:
        client.close()


sample_source = read_mongo_source_rows(["customers", "accounts", "fixed_deposit_plans"])
sample_source

# Part 3 - Generate MongoDB RDI pipeline files

The generated RDI files cover every Radish Bank entity. Each job writes Redis JSON with the same key template used by the domain's generated `ContextModel` class.

In [ ]:
import re
from textwrap import dedent

RDI_PIPELINE_DIR = NOTEBOOKS_DIR / "generated_rdi_pipeline" / "radish-bank-mongodb-local"
JOBS_DIR = RDI_PIPELINE_DIR / "jobs"


def _jmespath_key_expression(template: str) -> str:
    parts: list[str] = []
    cursor = 0
    for match in re.finditer(r"\{([^{}]+)\}", template):
        literal = template[cursor:match.start()]
        if literal:
            parts.append(repr(literal))
        parts.append(match.group(1))
        cursor = match.end()
    if cursor < len(template):
        parts.append(repr(template[cursor:]))
    return f"concat([{', '.join(parts)}])"


def _mongodb_config_yaml() -> str:
    tables = "\n".join(f"          {collection_name}: {{}}" for collection_name in records_by_collection)
    return dedent(
        f"""
        sources:
          radish-source:
            name: Local Radish Bank MongoDB source
            type: cdc
            connection:
              type: mongodb
              connection_string: ${{MONGODB_CONNECTION_STRING}}
              database: ${{MONGODB_DATABASE}}
            databases:
              - ${{MONGODB_DATABASE}}
            tables:
{tables}
        targets:
          target:
            name: Redis Iris context target
            connection:
              type: redis
              host: ${{REDIS_HOST}}
              port: ${{REDIS_PORT}}
              user: ${{REDIS_USERNAME}}
              password: ${{REDIS_PASSWORD}}
        processors:
          target_data_type: json
        metadata:
          name: radish-bank-local-mongodb-to-iris
          description: Local MongoDB collections synced to Redis JSON keys consumed by Redis Iris Context Retriever.
        """
    ).strip() + "\n"


def _job_yaml(collection_name: str, key_expr: str) -> str:
    return dedent(
        f"""
        name: Write {collection_name} to Redis JSON for Radish Bank Redis Iris
        source:
          table: {collection_name}
          row_format: partial
        output:
          - uses: redis.write
            with:
              connection: target
              data_type: json
              key:
                expression: {key_expr}
                language: jmespath
              on_update: replace
        """
    ).strip() + "\n"


JOBS_DIR.mkdir(parents=True, exist_ok=True)
(RDI_PIPELINE_DIR / "config.yaml").write_text(_mongodb_config_yaml(), encoding="utf-8")
for collection_name, key_template in key_template_by_collection.items():
    job_path = JOBS_DIR / f"{collection_name}.yaml"
    job_path.write_text(_job_yaml(collection_name, _jmespath_key_expression(key_template)), encoding="utf-8")

print(f"Generated RDI pipeline files under: {RDI_PIPELINE_DIR}")
for path in [RDI_PIPELINE_DIR / "config.yaml", *sorted(JOBS_DIR.glob("*.yaml"))]:
    print(f"  - {path.relative_to(ROOT)}")

In [ ]:
print("--- config.yaml ---")
print((RDI_PIPELINE_DIR / "config.yaml").read_text())
print("--- jobs/accounts.yaml ---")
print((JOBS_DIR / "accounts.yaml").read_text())

In [ ]:
# Values to provide to your RDI deployment through its normal secret/env mechanism.
RDI_ENV_HINTS = {
    "MONGODB_CONNECTION_STRING": MONGODB_URI,
    "MONGODB_DATABASE": MONGODB_DATABASE,
    "REDIS_HOST": os.environ.get("REDIS_HOST", ""),
    "REDIS_PORT": os.environ.get("REDIS_PORT", ""),
    "REDIS_USERNAME": os.environ.get("REDIS_USERNAME", "default"),
    "REDIS_PASSWORD": "<same Redis password from WORKSHOP_CONFIG>",
}
RDI_ENV_HINTS

## Deploy RDI outside the notebook

Use `notebooks/generated_rdi_pipeline/radish-bank-mongodb-local/` with your RDI deployment.

Checklist:

- Local MongoDB is running as replica set `rs0` and contains the seeded Radish Bank collections.
- The RDI runtime can reach MongoDB at `MONGODB_CONNECTION_STRING`. If RDI runs in another container or cluster, replace `localhost` with a host/IP reachable from that runtime.
- Target Redis is the same Redis database used by the Context Surface below.
- Generated job key expressions match the Radish Bank `ContextModel.__redis_key_template__` values.

Example CLI shape, if your environment uses `redis-di`:

```bash
redis-di deploy --dir notebooks/generated_rdi_pipeline/radish-bank-mongodb-local
```

For a workshop without a live RDI deployment, use the notebook lab lane below. It copies MongoDB records into Redis JSON using the same keys that RDI would write, so the Context Retriever and UI behavior are the same after data reaches Redis.

# Part 4 - Create the Redis Iris Context Surface

This creates or updates the Context Surface from the Radish Bank generated model classes. It requires Redis Cloud and Context Retriever credentials in `WORKSHOP_CONFIG`.

If your goal is the demo UI, this is the key handoff point: the UI backend lists tools from the `MCP_AGENT_KEY` created here and reads the active domain from `DEMO_DOMAIN=radish-bank`.

In [ ]:
require_iris_config()
apply_notebook_config(WORKSHOP_CONFIG)

import httpx
from context_surfaces import config as cs_config
from context_surfaces.context_model import export_data_model
from backend.app.settings import get_settings


def _admin_headers(admin_key: str) -> dict[str, str]:
    return {"Content-Type": "application/json", "X-API-Key": admin_key}


def _context_entities() -> list[type]:
    return [model_by_class[spec.class_name] for spec in entity_specs]


def _build_data_model() -> dict:
    return export_data_model(
        title=domain.manifest.namespace.surface_name,
        description=domain.manifest.description,
        entities=_context_entities(),
    )


async def create_or_update_surface(*, force_recreate: bool = False, update_existing: bool = True) -> tuple[str, str]:
    settings = get_settings()
    data_model = _build_data_model()
    api_url = str(cs_config.api_url).rstrip("/")
    surface_id = "" if force_recreate else os.environ.get("CTX_SURFACE_ID", "")
    agent_key = "" if force_recreate else os.environ.get("MCP_AGENT_KEY", "")

    body = {
        "name": domain.manifest.namespace.surface_name,
        "description": domain.manifest.description,
        "data_model": data_model,
        "data_source": {
            "type": "redis",
            "connection_config": {
                "addr": f"{settings.redis_host}:{settings.redis_port}",
                "username": settings.redis_username or "default",
                "password": settings.redis_password,
                "db": settings.redis_db,
                "tls_enabled": settings.redis_ssl,
                "pool_size": 10,
                "min_idle_conns": 2,
            },
        },
    }

    async with httpx.AsyncClient(timeout=60.0) as client:
        if surface_id and update_existing:
            response = await client.put(
                f"{api_url}/api/v1/context-surfaces/{surface_id}",
                headers=_admin_headers(settings.ctx_admin_key),
                json={"name": body["name"], "description": body["description"], "data_model": data_model},
            )
            response.raise_for_status()
        elif not surface_id:
            response = await client.post(
                f"{api_url}/api/v1/context-surfaces",
                headers=_admin_headers(settings.ctx_admin_key),
                json=body,
            )
            if response.status_code != 201:
                raise RuntimeError(f"Failed to create context surface ({response.status_code}): {response.text}")
            surface_id = str(response.json()["id"])

        if not agent_key:
            response = await client.post(
                f"{api_url}/api/v1/context-surfaces/{surface_id}/agent-keys",
                headers=_admin_headers(settings.ctx_admin_key),
                json={"name": domain.manifest.namespace.agent_name},
            )
            if response.status_code != 201:
                raise RuntimeError(f"Failed to create agent key ({response.status_code}): {response.text}")
            agent_key = str(response.json()["key"])

    os.environ["CTX_SURFACE_ID"] = surface_id
    os.environ["MCP_AGENT_KEY"] = agent_key
    return surface_id, agent_key


surface_id, agent_key = await create_or_update_surface(update_existing=True)
settings = get_settings()
print("Surface:", surface_id)
print("Agent key:", "SET" if agent_key else "MISSING")

# Part 5 - Notebook lab lane: copy MongoDB to Redis JSON

Skip this if real RDI already wrote the target keys. Otherwise, this cell reads local MongoDB and writes the same Redis JSON documents that the generated RDI jobs would produce.

After this cell succeeds, Redis contains the `radish_bank_*` JSON records needed by both Context Retriever and the demo UI.

In [ ]:
import json
from redis.asyncio import Redis


def _redis_json_client(settings):
    return Redis(
        host=settings.redis_host,
        port=settings.redis_port,
        username=settings.redis_username or "default",
        password=settings.redis_password,
        db=settings.redis_db,
        ssl=settings.redis_ssl,
        decode_responses=True,
    )


def _redis_key(collection_name: str, row: dict) -> str:
    return key_template_by_collection[collection_name].format(**row)


async def write_rdi_target_snapshot(collections: list[str] | None = None) -> dict[str, int]:
    """Lab-mode RDI substitute: copy local Mongo rows into Redis JSON keys."""
    source_rows = read_mongo_source_rows(collections)
    client = _redis_json_client(settings)
    counts: dict[str, int] = {}
    try:
        for collection_name, rows in source_rows.items():
            counts[collection_name] = 0
            for row in rows:
                key = _redis_key(collection_name, row)
                await client.execute_command("JSON.SET", key, "$", json.dumps(row))
                counts[collection_name] += 1
    finally:
        await client.aclose()
    return counts


redis_counts = await write_rdi_target_snapshot()
summary = domain.write_dataset_meta(settings=settings, records=records_by_class)
print("Redis JSON writes:", redis_counts)
print("Dataset meta:", summary)

# Part 6 - Verify Context Retriever

First inspect Redis JSON directly, then call Context Retriever MCP tools generated from the Radish Bank data model.

In [ ]:
async def json_get(key: str):
    client = _redis_json_client(settings)
    try:
        raw = await client.execute_command("JSON.GET", key, "$")
    finally:
        await client.aclose()
    if not raw:
        return None
    value = json.loads(raw)
    return value[0] if isinstance(value, list) else value


await json_get("radish_bank_account:ACC001")

In [ ]:
from backend.app.context_surface_service import ContextSurfaceService

cs_service = ContextSurfaceService(settings)
tools = await cs_service.list_tools()
print(f"{len(tools)} Context Retriever tools")
for tool in tools[:25]:
    print(f"  - {tool['name']}")

In [ ]:
accounts = await cs_service.call_tool(
    "filter_account_by_customer_id",
    {"value": os.environ.get("DEMO_USER_ID", "CUST001")},
)
accounts

## Freshness test

Update local MongoDB. In the real RDI lane, change streams propagate the update. In the notebook lab lane, run the snapshot writer for `accounts` again. Context Retriever then sees fresh data with no model or agent-code change.

In [ ]:
def update_mongo_account_balance(account_id: str, balance_sgd: float) -> dict:
    client = mongo_client()
    try:
        db = client[MONGODB_DATABASE]
        result = db.accounts.update_one(
            {"account_id": account_id},
            {"$set": {"balance_sgd": balance_sgd}},
        )
        if result.matched_count != 1:
            raise RuntimeError(f"No account found for {account_id}")
        row = db.accounts.find_one({"account_id": account_id}, {"_id": 0})
        return dict(row)
    finally:
        client.close()


updated_account = update_mongo_account_balance("ACC001", 54500.0)
print("MongoDB source row:", updated_account)

await write_rdi_target_snapshot(["accounts"])
print("Redis JSON row:", await json_get("radish_bank_account:ACC001"))

fresh_accounts = await cs_service.call_tool(
    "filter_account_by_customer_id",
    {"value": os.environ.get("DEMO_USER_ID", "CUST001")},
)
fresh_accounts

# Part 7 - Seed Agent Memory and LangCache

These seeds come from the Radish Bank domain manifest.

In [ ]:
from backend.app.memory_service import MemoryService
from backend.app.langcache_service import LangCacheService

memory_service = MemoryService(settings, similarity_threshold=0.5)
langcache_service = LangCacheService(settings)
owner_id = os.environ.get("DEMO_USER_ID", domain.manifest.identity.default_id)

for index, memory in enumerate(domain.manifest.seed_memories, start=1):
    memory_service.create_long_term_memory(
        text=memory.text,
        owner_id=owner_id,
        memory_type=memory.memory_type,
        topics=memory.topics,
        memory_id=f"radish-workshop-seed-{index}",
    )

print(f"Seeded {len(domain.manifest.seed_memories)} Radish Bank long-term memories")

In [ ]:
memories = await memory_service.asearch_long_term_memory(
    text="fixed deposit funding account preferences",
    owner_id=owner_id,
    limit=5,
)
for item in memories:
    print(f"- [{item.get('id', '?')}] {item.get('text', item)}")

In [ ]:
for entry in domain.manifest.seed_langcache:
    ok = await langcache_service.store(entry.prompt, entry.response, attributes=entry.attributes)
    print(f"LangCache seed for {entry.prompt!r}: {'OK' if ok else 'FAILED'}")

for question in [
    domain.manifest.seed_langcache[0].prompt,
    "Tell me about your FD rates",
    "What is my savings account balance?",
]:
    result = await langcache_service.search(question)
    print(f"{'HIT' if result else 'MISS'}: {question}", f"({result['similarity']:.3f})" if result else "")

# Part 8 - Domain-aware chat loop

This uses the same backend agent construction path as the app: Radish Bank internal tools plus Context Retriever MCP tools, with LangCache and memory enrichment in front.

In [ ]:
from backend.app.internal_tools import InternalToolService
from backend.app.langgraph_agent import create_agent
from langchain_core.messages import HumanMessage

internal_tools = InternalToolService(settings)
agent = await create_agent(settings, internal_tools, cs_service, checkpointer=None)
SESSION_ID = f"radish-bank-workshop-{os.urandom(4).hex()}"


def _memory_text(items: list[dict]) -> str:
    lines = []
    for item in items[:5]:
        text = str(item.get("text", "")).strip()
        if text:
            lines.append(f"- {text}")
    return "\n".join(lines)


async def radish_chat_turn(message: str) -> dict:
    trace: list[str] = []
    user_message = message.strip()
    if not user_message:
        raise ValueError("message must not be empty")

    cache_result = await langcache_service.search(user_message)
    if cache_result:
        trace.append(f"LangCache HIT ({cache_result.get('similarity', 0):.3f})")
        return {"assistant": str(cache_result.get("response", "")), "trace": trace, "cache_hit": True}
    trace.append("LangCache MISS")

    if memory_service.is_configured():
        await memory_service.add_session_event(
            owner_id=owner_id,
            session_id=SESSION_ID,
            actor_id=owner_id,
            role="USER",
            text=user_message,
            metadata={"source": "radish-bank-rdi-workshop"},
        )
        memories = await memory_service.asearch_long_term_memory(text=user_message, owner_id=owner_id, limit=5)
        trace.append(f"Long-term memory matches: {len(memories)}")
    else:
        memories = []

    enriched_message = user_message
    memory_context = _memory_text(memories)
    if memory_context:
        enriched_message = f"{user_message}\n\nRetrieved memory context:\n{memory_context}"

    final_text = ""
    async for event in agent.astream_events(
        {"messages": [HumanMessage(content=enriched_message)]},
        config={"configurable": {"thread_id": SESSION_ID}},
        version="v2",
    ):
        kind = event["event"]
        if kind == "on_tool_start":
            trace.append(f"Tool call: {event.get('name', '')}")
        elif kind == "on_chat_model_stream":
            chunk = event["data"].get("chunk")
            if chunk and getattr(chunk, "content", None) and not getattr(chunk, "tool_calls", None):
                final_text += chunk.content

    if not final_text.strip() and hasattr(agent, "aget_state"):
        snapshot = await agent.aget_state({"configurable": {"thread_id": SESSION_ID}})
        messages = snapshot.values.get("messages", []) if snapshot and snapshot.values else []
        for msg in reversed(messages):
            if getattr(msg, "content", None) and not getattr(msg, "tool_calls", None):
                final_text = msg.content if isinstance(msg.content, str) else str(msg.content)
                break

    final_text = final_text.strip()
    if memory_service.is_configured() and final_text:
        await memory_service.add_session_event(
            owner_id=owner_id,
            session_id=SESSION_ID,
            actor_id="assistant",
            role="ASSISTANT",
            text=final_text,
            metadata={"source": "radish-bank-rdi-workshop"},
        )

    return {"assistant": final_text, "trace": trace, "cache_hit": False}


print("Session:", SESSION_ID)

In [ ]:
turn1 = await radish_chat_turn("Tell me about your fixed deposit interest rates")
print("--- trace ---")
print("\n".join(turn1["trace"]))
print("--- assistant ---")
print(turn1["assistant"])

In [ ]:
turn2 = await radish_chat_turn("What accounts do I have and what are my current balances?")
print("--- trace ---")
print("\n".join(turn2["trace"]))
print("--- assistant ---")
print(turn2["assistant"])

In [ ]:
turn3 = await radish_chat_turn("Given what you know about my preferences, can I place SGD 5,000 into the 6-month fixed deposit?")
print("--- trace ---")
print("\n".join(turn3["trace"]))
print("--- assistant ---")
print(turn3["assistant"])

# Optional - Run The Demo UI

After the Context Surface exists and Redis contains the Radish Bank JSON records, run the branded app from a terminal:

```bash
# Put the same WORKSHOP_CONFIG values into .env first, including:
# DEMO_DOMAIN=radish-bank
# CTX_SURFACE_ID=<surface id from this notebook>
# MCP_AGENT_KEY=<agent key from this notebook>
make dev
```

Open the frontend URL printed by `make dev`. The UI will use the existing Radish Bank branding, starter prompts, Memory panel, tool trace panel, LangCache, and Context Retriever tools.